# Pose training notebook

Cleaned version of the original training notebook.

The main change is a single `FOREST_ENABLED` switch. When it is `False`, forest data is not loaded, forest batches are not processed, forest metrics are not logged, and forest plots are not created.


In [23]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from dataset import SceneTwoPairsDataset
from colmap_pair_dataset import ColmapPairDataset
from model import PairImageCylinderModel
from losses import (
    supervised_loss,
    matched_radius_consistency_loss,
    matched_reprojection_loss_2d,
)
from debugger import bug_check, debug_sinkhorn_matching
from utils import plot_estimated_cylinders_on_images, plot_relative_pose, pose_to_text


In [24]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

BATCH_SIZE = 16
NUM_EPOCHS = 30
VAL_INTERVAL = 10
LEARNING_RATE = 5e-4

# Checkpoint settings
BEST_MODEL_PATH = "best_model.pt"
# Choose which trained weights to use for test/inference: "best" or "latest".
INFERENCE_MODEL = "best"

# Forest controls
FOREST_ENABLED = False
FOREST_START_EPOCH = 101
USE_FOREST_IN_TOTAL_LOSS = False
FOREST_LOSS_WEIGHT = 1.0

# Loss settings
LAMBDA_OCC = 10.0
LAMBDA_RADIUS = 10.0
LAMBDA_RAD = 10.0
LAMBDA_REPROJ = 1.0
LAMBDA_FOREST_SPARSITY = 0.1
OCC_THRESH = 0.5

# Visualization settings
INFERENCE_BATCH_SIZE = 4
INFERENCE_SHOW_TEXT = False
INFERENCE_SHOW_IMAGES = False
FOREST_SAMPLE_IDX = 0
FOREST_OCC_THRESHOLD = 0.3
VISION_SAMPLE_IDX = 0

assert not USE_FOREST_IN_TOTAL_LOSS or FOREST_ENABLED, (
    "USE_FOREST_IN_TOTAL_LOSS requires FOREST_ENABLED=True"
)


In [25]:
# Device and model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

model = PairImageCylinderModel(
    img_size=128,
    patch_size=8,
    in_chans=3,
    embed_dim=384,
    depth=6,
    num_heads=6,
    num_bins=128,
    dropout=0.1,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


CUDA available: True
NVIDIA GeForce RTX 5050 Laptop GPU


## Data

Training and validation data are always available. The forest loader is created only when `FOREST_ENABLED=True`.


In [26]:
train_dataset = SceneTwoPairsDataset(
    root_dir="dataset",
    image_size=128,
    debug=False,
    return_two_pairs=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_dataset = SceneTwoPairsDataset(
    root_dir="valdataset",
    image_size=128,
    debug=False,
    return_two_pairs=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

forest_loader = None
if FOREST_ENABLED:
    forest_dataset = ColmapPairDataset(
        dataset_dir="forestdataset",
        image_size=128,
    )
    forest_loader = DataLoader(
        forest_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
if FOREST_ENABLED:
    print("Forest batches:", len(forest_loader))


Train batches: 36
Val batches: 7


## Optional debug checks

Run these only when needed. They do not affect training.


In [27]:
# bug_check(0)
# debug_sinkhorn_matching(device=device)


In [28]:
# -----------------------------------------------------------------------------
# Training helpers
# -----------------------------------------------------------------------------

def compute_supervised_pair_losses(
    pred_vision_a1,
    vision_a1,
    pred_vision_b1,
    vision_b1,
    pred_pose1,
    pose_ab1,
    pred_vision_a2,
    vision_a2,
    pred_vision_b2,
    vision_b2,
    pred_pose2,
    pose_ab2,
):
    loss_out1 = supervised_loss(
        pred_vision_a1,
        vision_a1,
        pred_vision_b1,
        vision_b1,
        pred_pose1,
        pose_ab1,
        occ_thresh=OCC_THRESH,
        lambda_occ=LAMBDA_OCC,
        lambda_radius=LAMBDA_RADIUS,
    )
    loss_out2 = supervised_loss(
        pred_vision_a2,
        vision_a2,
        pred_vision_b2,
        vision_b2,
        pred_pose2,
        pose_ab2,
        occ_thresh=OCC_THRESH,
        lambda_occ=LAMBDA_OCC,
        lambda_radius=LAMBDA_RADIUS,
    )

    radius_cons1 = matched_radius_consistency_loss(
        pred_vision_a1,
        vision_a1,
        pred_vision_b1,
        vision_b1,
        relative_pose_pred=pred_pose1,
        matching_mode="gt",
        occ_thresh=OCC_THRESH,
    )
    radius_cons2 = matched_radius_consistency_loss(
        pred_vision_a2,
        vision_a2,
        pred_vision_b2,
        vision_b2,
        relative_pose_pred=pred_pose2,
        matching_mode="gt",
        occ_thresh=OCC_THRESH,
    )

    reproj1 = matched_reprojection_loss_2d(
        pred_vision_a1,
        vision_a1,
        pred_vision_b1,
        vision_b1,
        pred_pose1,
        matching_mode="gt",
        occ_thresh=OCC_THRESH,
        fov_degrees=90.0,
    )
    reproj2 = matched_reprojection_loss_2d(
        pred_vision_a2,
        vision_a2,
        pred_vision_b2,
        vision_b2,
        pred_pose2,
        matching_mode="gt",
        occ_thresh=OCC_THRESH,
        fov_degrees=90.0,
    )

    return (
        loss_out1,
        loss_out2,
        (radius_cons1 + radius_cons2) / 2.0,
        (reproj1 + reproj2) / 2.0,
    )


def compute_forest_loss(forest_batch):
    img_a_forest, img_b_forest, _ = tuple(
        x.to(device, non_blocking=True) for x in forest_batch
    )

    pred_vision_a_forest, pred_vision_b_forest, pred_pose_forest = model(
        img_a_forest,
        img_b_forest,
    )

    forest_radius_cons = matched_radius_consistency_loss(
        pred_vision_a_forest,
        pred_vision_b=pred_vision_b_forest,
        relative_pose_pred=pred_pose_forest,
        matching_mode="sinkhorn",
        occ_thresh=OCC_THRESH,
    )
    forest_reproj = matched_reprojection_loss_2d(
        pred_vision_a_forest,
        pred_vision_b=pred_vision_b_forest,
        relative_pose_pred=pred_pose_forest,
        matching_mode="sinkhorn",
        occ_thresh=OCC_THRESH,
        fov_degrees=90.0,
    )
    forest_sparsity = (
        pred_vision_a_forest[..., 0].mean()
        + pred_vision_b_forest[..., 0].mean()
    ) / 2.0

    forest_loss = (
        LAMBDA_RAD * forest_radius_cons
        + LAMBDA_REPROJ * forest_reproj
        + LAMBDA_FOREST_SPARSITY * forest_sparsity
    )

    return {
        "total": forest_loss,
        "radius_consistency": forest_radius_cons,
        "reprojection": forest_reproj,
        "sparsity": forest_sparsity,
    }


def validate():
    model.eval()
    totals = {
        "total": 0.0,
        "supervised": 0.0,
        "vision": 0.0,
        "pose": 0.0,
        "radius_consistency": 0.0,
        "reprojection": 0.0,
    }

    num_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            img_a, vision_a, img_b, vision_b, pose_ab = tuple(
                x.to(device, non_blocking=True) for x in batch
            )

            pred_vision_a, pred_vision_b, pred_pose = model(img_a, img_b)

            loss_out = supervised_loss(
                pred_vision_a,
                vision_a,
                pred_vision_b,
                vision_b,
                pred_pose,
                pose_ab,
                occ_thresh=OCC_THRESH,
                lambda_occ=LAMBDA_OCC,
                lambda_radius=LAMBDA_RADIUS,
            )

            radius_cons = matched_radius_consistency_loss(
                pred_vision_a,
                vision_a,
                pred_vision_b,
                vision_b,
                relative_pose_pred=pred_pose,
                matching_mode="gt",
                occ_thresh=OCC_THRESH,
            )
            reproj = matched_reprojection_loss_2d(
                pred_vision_a,
                vision_a,
                pred_vision_b,
                vision_b,
                pred_pose,
                matching_mode="gt",
                occ_thresh=OCC_THRESH,
                fov_degrees=90.0,
            )

            sup_loss = loss_out["total"]
            totals["total"] += sup_loss.item()
            totals["supervised"] += sup_loss.item()
            totals["vision"] += loss_out["vision_a"].item()
            totals["pose"] += loss_out["pose"].item()
            totals["radius_consistency"] += radius_cons.item()
            totals["reprojection"] += reproj.item()
            num_batches += 1

    if num_batches == 0:
        raise RuntimeError("Validation loader is empty; cannot select a best model.")

    return {key: value / num_batches for key, value in totals.items()}


## Training

Forest metrics are created only while forest is active. No forest values are appended to history and no forest values are included in the log otherwise.

The default keeps the original behavior: `loss = supervised_loss`. Set `USE_FOREST_IN_TOTAL_LOSS=True` to actually add the forest objective to the optimization loss.


In [ ]:
history = {
    "epoch": [],
    "total": [],
    "supervised": [],
    "vision": [],
    "pose": [],
    "radius_consistency": [],
    "reprojection": [],
    "forest_epoch": [],
    "forest_total": [],
    "forest_radius_consistency": [],
    "forest_reprojection": [],
    "forest_sparsity": [],
    "val_epoch": [],
    "val_total": [],
    "val_supervised": [],
    "val_vision": [],
    "val_pose": [],
    "val_radius_consistency": [],
    "val_reprojection": [],
}

best_val_loss = float("inf")
best_epoch = None

for epoch in range(NUM_EPOCHS):
    epoch_num = epoch + 1
    model.train()

    forest_active = (
        FOREST_ENABLED
        and forest_loader is not None
        and epoch_num >= FOREST_START_EPOCH
    )
    forest_iter = iter(forest_loader) if forest_active else None

    train_totals = {
        "total": 0.0,
        "supervised": 0.0,
        "vision": 0.0,
        "pose": 0.0,
        "radius_consistency": 0.0,
        "reprojection": 0.0,
    }

    forest_totals = {
        "total": 0.0,
        "radius_consistency": 0.0,
        "reprojection": 0.0,
        "sparsity": 0.0,
    }

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)

        (
            img_a1,
            vision_a1,
            img_b1,
            vision_b1,
            pose_ab1,
            img_a2,
            vision_a2,
            img_b2,
            vision_b2,
            pose_ab2,
        ) = tuple(x.to(device, non_blocking=True) for x in batch)

        (
            pred_vision_a1,
            pred_vision_b1,
            pred_pose1,
        ) = model(img_a1, img_b1)
        (
            pred_vision_a2,
            pred_vision_b2,
            pred_pose2,
        ) = model(img_a2, img_b2)

        loss_out1, loss_out2, radius_cons, reproj = compute_supervised_pair_losses(
            pred_vision_a1,
            vision_a1,
            pred_vision_b1,
            vision_b1,
            pred_pose1,
            pose_ab1,
            pred_vision_a2,
            vision_a2,
            pred_vision_b2,
            vision_b2,
            pred_pose2,
            pose_ab2,
        )

        sup_loss = (loss_out1["total"] + loss_out2["total"]) / 2.0
        loss = sup_loss

        if forest_active:
            try:
                batch_forest = next(forest_iter)
            except StopIteration:
                forest_iter = iter(forest_loader)
                batch_forest = next(forest_iter)

            forest_metrics_batch = compute_forest_loss(batch_forest)

            for key, value in forest_metrics_batch.items():
                forest_totals[key] += value.item()

            if USE_FOREST_IN_TOTAL_LOSS:
                loss = loss + FOREST_LOSS_WEIGHT * forest_metrics_batch["total"]

        loss.backward()
        optimizer.step()

        train_totals["total"] += loss.item()
        train_totals["supervised"] += sup_loss.item()
        train_totals["vision"] += (
            (loss_out1["vision_a"] + loss_out2["vision_a"]) / 2.0
        ).item()
        train_totals["pose"] += (
            (loss_out1["pose"] + loss_out2["pose"]) / 2.0
        ).item()
        train_totals["radius_consistency"] += radius_cons.item()
        train_totals["reprojection"] += reproj.item()

    train_metrics = {
        key: value / len(train_loader)
        for key, value in train_totals.items()
    }

    history["epoch"].append(epoch_num)
    history["total"].append(train_metrics["total"])
    history["supervised"].append(train_metrics["supervised"])
    history["vision"].append(train_metrics["vision"])
    history["pose"].append(train_metrics["pose"])
    history["radius_consistency"].append(train_metrics["radius_consistency"])
    history["reprojection"].append(train_metrics["reprojection"])

    log = (
        f"Epoch {epoch_num}: "
        f"tot={train_metrics['total']:.4f} | "
        f"sup={train_metrics['supervised']:.4f} | "
        f"vis={train_metrics['vision']:.4f} | "
        f"pose={train_metrics['pose']:.4f} | "
        f"radius_cons={train_metrics['radius_consistency']:.4f} | "
        f"reproj={train_metrics['reprojection']:.4f}"
    )

    if forest_active:
        forest_metrics = {
            key: value / len(train_loader)
            for key, value in forest_totals.items()
        }
        history["forest_epoch"].append(epoch_num)
        history["forest_total"].append(forest_metrics["total"])
        history["forest_radius_consistency"].append(
            forest_metrics["radius_consistency"]
        )
        history["forest_reprojection"].append(forest_metrics["reprojection"])
        history["forest_sparsity"].append(forest_metrics["sparsity"])

        log += (
            f" | forest_tot={forest_metrics['total']:.4f} | "
            f"forest_radius_cons={forest_metrics['radius_consistency']:.4f} | "
            f"forest_reproj={forest_metrics['reprojection']:.4f} | "
            f"forest_sparse={forest_metrics['sparsity']:.4f}"
        )

    if epoch_num == 1 or epoch_num % VAL_INTERVAL == 0:
        val_metrics = validate()
        history["val_epoch"].append(epoch_num)
        history["val_total"].append(val_metrics["total"])
        history["val_supervised"].append(val_metrics["supervised"])
        history["val_vision"].append(val_metrics["vision"])
        history["val_pose"].append(val_metrics["pose"])
        history["val_radius_consistency"].append(
            val_metrics["radius_consistency"]
        )
        history["val_reprojection"].append(val_metrics["reprojection"])

        is_best = val_metrics["total"] < best_val_loss
        if is_best:
            best_val_loss = val_metrics["total"]
            best_epoch = epoch_num
            torch.save(
                {
                    "epoch": epoch_num,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_metrics": val_metrics,
                    "best_val_loss": best_val_loss,
                    "history": history,
                },
                BEST_MODEL_PATH,
            )

        log += (
            f" | val_tot={val_metrics['total']:.4f} | "
            f"val_sup={val_metrics['supervised']:.4f} | "
            f"val_vis={val_metrics['vision']:.4f} | "
            f"val_pose={val_metrics['pose']:.4f} | "
            f"val_radius_cons={val_metrics['radius_consistency']:.4f} | "
            f"val_reproj={val_metrics['reprojection']:.4f}"
        )

        if is_best:
            log += " | BEST MODEL SAVED"

    print(log)


# Keep a copy of the final epoch for optional test/inference.
latest_model_state_dict = {
    key: value.detach().cpu().clone()
    for key, value in model.state_dict().items()
}

print(f"Best validation loss: {best_val_loss:.6f} at epoch {best_epoch}")
print(f"Best model saved to: {BEST_MODEL_PATH}")


Epoch 1: tot=16.1207 | sup=16.1207 | vis=15.8069 | pose=0.2803 | radius_cons=0.0066 | reproj=0.0056 | val_tot=9.9722 | val_sup=9.9722 | val_vis=9.6998 | val_pose=0.2540 | val_radius_cons=0.0000 | val_reproj=0.0000 | BEST MODEL SAVED
Epoch 2: tot=9.7763 | sup=9.7763 | vis=9.5004 | pose=0.2611 | radius_cons=0.0000 | reproj=0.0000
Epoch 3: tot=9.6202 | sup=9.6202 | vis=9.3507 | pose=0.2631 | radius_cons=0.0000 | reproj=0.0000
Epoch 4: tot=9.6447 | sup=9.6447 | vis=9.3644 | pose=0.2602 | radius_cons=0.0000 | reproj=0.0000
Epoch 5: tot=9.6505 | sup=9.6505 | vis=9.3779 | pose=0.2596 | radius_cons=0.0000 | reproj=0.0000
Epoch 6: tot=9.5990 | sup=9.5990 | vis=9.3303 | pose=0.2616 | radius_cons=0.0000 | reproj=0.0000
Epoch 7: tot=9.6635 | sup=9.6635 | vis=9.4004 | pose=0.2596 | radius_cons=0.0000 | reproj=0.0000
Epoch 8: tot=9.5977 | sup=9.5977 | vis=9.3289 | pose=0.2607 | radius_cons=0.0000 | reproj=0.0000
Epoch 9: tot=9.5824 | sup=9.5824 | vis=9.3121 | pose=0.2618 | radius_cons=0.0000 | repro

## Load best validation model

Before test inference, load the checkpoint selected using the lowest validation `total` loss.


In [ ]:
# No model is loaded here. The inference cell below chooses "best" or "latest".


## Training plots

In [ ]:
epochs = np.array(history["epoch"])
val_epochs = np.array(history["val_epoch"])
forest_epochs = np.array(history["forest_epoch"])

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["supervised"], label="train supervised")
plt.plot(epochs, history["reprojection"], label="train reprojection")
plt.plot(epochs, history["total"], label="train total")

if FOREST_ENABLED and len(forest_epochs) > 0:
    plt.plot(forest_epochs, history["forest_total"], "--", label="forest total")

if len(val_epochs) > 0:
    plt.plot(val_epochs, history["val_supervised"], ".--", label="val supervised")
    plt.plot(val_epochs, history["val_reprojection"], ".--", label="val reprojection")
    plt.plot(val_epochs, history["val_total"], ".--", label="val total")

if FOREST_ENABLED:
    plt.axvline(FOREST_START_EPOCH, linestyle=":", alpha=0.5, label="forest start")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["radius_consistency"], label="train radius_consistency")

if FOREST_ENABLED and len(forest_epochs) > 0:
    plt.plot(
        forest_epochs,
        history["forest_radius_consistency"],
        "--",
        label="forest radius_consistency",
    )

if len(val_epochs) > 0:
    plt.plot(
        val_epochs,
        history["val_radius_consistency"],
        ".--",
        label="val radius_consistency",
    )

if FOREST_ENABLED:
    plt.axvline(FOREST_START_EPOCH, linestyle=":", alpha=0.5, label="forest start")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Radius consistency loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["reprojection"], label="train reprojection")

if FOREST_ENABLED and len(forest_epochs) > 0:
    plt.plot(
        forest_epochs,
        history["forest_reprojection"],
        "--",
        label="forest reprojection",
    )

if len(val_epochs) > 0:
    plt.plot(
        val_epochs,
        history["val_reprojection"],
        ".--",
        label="val reprojection",
    )

if FOREST_ENABLED:
    plt.axvline(FOREST_START_EPOCH, linestyle=":", alpha=0.5, label="forest start")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Reprojection loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Inference on test data

In [ ]:
# Choose the model used for test/inference:
#   "best"   -> checkpoint with lowest validation loss
#   "latest" -> weights from the final training epoch
# This is independent of whether the best checkpoint is loaded later.
INFERENCE_MODEL = INFERENCE_MODEL.lower()
if INFERENCE_MODEL not in {"best", "latest"}:
    raise ValueError("INFERENCE_MODEL must be either 'best' or 'latest'")

if INFERENCE_MODEL == "best":
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(
        f"Using BEST model from epoch {checkpoint['epoch']} "
        f"with val_total={checkpoint['best_val_loss']:.6f}"
    )
else:
    model.load_state_dict(latest_model_state_dict)
    print(f"Using LATEST model from epoch {NUM_EPOCHS}")

model.eval()

infer_dataset = SceneTwoPairsDataset(
    root_dir="testdataset",
    return_two_pairs=False,
)
infer_loader = DataLoader(
    infer_dataset,
    batch_size=INFERENCE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

batch = next(iter(infer_loader))
img_a, vision_a, img_b, vision_b, pose_ab = batch

img_a = img_a.to(device)
vision_a = vision_a.to(device)
img_b = img_b.to(device)
vision_b = vision_b.to(device)
pose_ab = pose_ab.to(device)

with torch.no_grad():
    pred_vision, pred_vision_b, pred_pose = model(img_a, img_b)

if INFERENCE_SHOW_TEXT:
    n_show = min(4, img_a.shape[0])
    for i in range(n_show):
        print(f"Sample {i}")
        print("  GT   pose:", pose_to_text(pose_ab[i].detach().cpu()))
        print("  Pred pose:", pose_to_text(pred_pose[i].detach().cpu()))
        pose_err = torch.abs(pred_pose[i] - pose_ab[i]).detach().cpu()
        print(
            f"  |err|: tx={pose_err[0]:.3f}, "
            f"ty={pose_err[1]:.3f}, "
            f"sin={pose_err[2]:.3f}, "
            f"cos={pose_err[3]:.3f}"
        )
        print()

print("pred_vision shape:", pred_vision.shape)
print("pred_pose shape:", pred_pose.shape)
print("GT pose:", pose_ab[VISION_SAMPLE_IDX].detach().cpu())
print("Pred pose:", pred_pose[VISION_SAMPLE_IDX].detach().cpu())


## Vision prediction plots

In [ ]:
sample_idx = VISION_SAMPLE_IDX
bins = np.arange(pred_vision.shape[1])
occ_threshold = OCC_THRESH

if INFERENCE_SHOW_IMAGES:
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    names = ["occupancy", "radius", "depth"]

    gt_occ = vision_a[sample_idx, :, 0].detach().cpu().numpy()
    pr_occ = pred_vision[sample_idx, :, 0].detach().cpu().numpy()

    for j, ax in enumerate(axes):
        gt = vision_a[sample_idx, :, j].detach().cpu().numpy()
        pr = pred_vision[sample_idx, :, j].detach().cpu().numpy()

        ax.plot(bins, gt, label="GT", color="tab:blue")

        if j == 0:
            ax.plot(bins, pr, label="Pred", color="tab:orange")
        else:
            pr_mask = pr_occ > occ_threshold
            ax.scatter(
                bins[pr_mask],
                pr[pr_mask],
                label="Pred",
                color="tab:orange",
                s=25,
            )

        ax.set_ylabel(names[j])
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[-1].set_xlabel("Bin")
    fig.suptitle(f"Vision prediction sample {sample_idx}")
    plt.tight_layout()
    plt.show()

    fig_cyl, axes_cyl = plot_estimated_cylinders_on_images(
        [img_a[sample_idx], img_b[sample_idx]],
        [pred_vision[sample_idx], pred_vision_b[sample_idx]],
        titles=["Image A: pred cylinders", "Image B: pred cylinders"],
        occ_threshold=occ_threshold,
        flip_x=True,
    )
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_a[sample_idx].cpu().permute(1, 2, 0))
    axes[0].set_title("Image A")
    axes[0].axis("off")
    axes[1].imshow(img_b[sample_idx].cpu().permute(1, 2, 0))
    axes[1].set_title("Image B")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    plot_relative_pose(pose_ab[sample_idx], pred_pose[sample_idx])


## Forest visualization

In [ ]:
if FOREST_ENABLED:
    forest_sample_idx = FOREST_SAMPLE_IDX
    model.eval()
    forest_batch = next(iter(forest_loader))
    img_a_forest, img_b_forest, _ = forest_batch

    img_a_forest = img_a_forest.to(device)
    img_b_forest = img_b_forest.to(device)

    with torch.no_grad():
        pred_vision_a_forest, pred_vision_b_forest, pred_pose_forest = model(
            img_a_forest,
            img_b_forest,
        )

    fig_forest_cyl, axes_forest_cyl = plot_estimated_cylinders_on_images(
        [
            img_a_forest[forest_sample_idx],
            img_b_forest[forest_sample_idx],
        ],
        [
            pred_vision_a_forest[forest_sample_idx],
            pred_vision_b_forest[forest_sample_idx],
        ],
        titles=["Forest A: pred cylinders", "Forest B: pred cylinders"],
        occ_threshold=FOREST_OCC_THRESHOLD,
        flip_x=True,
    )
    plt.show()

    print("forest pred_vision_a shape:", pred_vision_a_forest.shape)
    print("forest pred_pose shape:", pred_pose_forest.shape)
    print(
        "forest pred_pose:",
        pred_pose_forest[forest_sample_idx].detach().cpu(),
    )


## Notes

- `FOREST_ENABLED=False` means no forest data is loaded and no forest log/plot is produced.
- `FOREST_ENABLED=True` enables the forest metrics from `FOREST_START_EPOCH` onward.
- `USE_FOREST_IN_TOTAL_LOSS=False` preserves the original optimization behavior.
- Set `USE_FOREST_IN_TOTAL_LOSS=True` when you want the forest objective to affect gradients.
- Validation runs on epoch 1 and every `VAL_INTERVAL` epochs. The checkpoint with the lowest validation `total` loss is saved to `BEST_MODEL_PATH`.
- Test/inference uses the best validation checkpoint when `LOAD_BEST_MODEL_FOR_INFERENCE=True`.
